# Cell Assembly Map Visualization

This notebook visualizes spatial protein co-occupancy maps for cell assemblies
annotated in the U2OS Cell Map, using ProtVS predictions as the protein image source.

**Workflow:**
1. Load assembly annotations and select high-scoring assemblies
2. Generate per-protein predictions with ProtVS and save to disk
3. Load predictions and build per-cell image stacks
4. Visualize protein tiles and co-occupancy heatmaps for each assembly
5. Export a standalone colorbar

## Setup

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import numpy as np
import os
from pathlib import Path

import pandas as pd
from skimage.filters import threshold_otsu
from tifffile import imread, imwrite
from tqdm.auto import tqdm

from protvs import ProtVS

## Configuration

In [ ]:
# Cell and model parameters
CELL_ID          = 1
CELL_LINE        = "U2OS"
TILE_SIZE        = 512          # native image crop (pixels)
DISPLAY_PX       = 256          # rendered tile size in the figure
N_REPRESENTATIVE = 5            # max protein tiles shown per assembly
THRESHOLD_MULT   = 1.5          # Otsu multiplier for binarization
DPI              = 300

# Font sizes
FONT_PROTEIN = 10
FONT_TITLE   = 12
FONT_CBAR    = 9

# Input paths — update these to match your local setup
DATA_DIR      = 'data'                                                         # root folder for input data
REF_PATH      = os.path.join(DATA_DIR, f'{CELL_LINE}_image_{CELL_ID}.tiff')   # reference cell image
ASSEMBLY_CSV  = os.path.join(DATA_DIR, 'cell_map_assemblies.csv')              # assembly annotation table

# Output directories
PRED_DIR = os.path.join('predictions', f'{CELL_LINE}_{CELL_ID:03d}')
OUT_DIR  = os.path.join('figures',     f'{CELL_LINE}_{CELL_ID:03d}')
os.makedirs(PRED_DIR, exist_ok=True)
os.makedirs(OUT_DIR,  exist_ok=True)

print(f"Cell: {CELL_LINE} #{CELL_ID}")
print(f"Predictions → {PRED_DIR}/")
print(f"Figures     → {OUT_DIR}/")

## Assembly Selection

Load the U2OS Cell Map CSV and collect all proteins that belong to
high-scoring assemblies — these are the proteins we need to predict.

In [ ]:
systems_data = pd.read_csv(ASSEMBLY_CSV)

# Keep only high-scoring assemblies with a name and protein list
high_score = systems_data[
    (systems_data.get('GPT4 score category') == 'High Score') &
    (systems_data['GPT4 default name'].notna()) &
    (systems_data['GPT4 default name'] != 'None') &
    (systems_data['Proteins'].notna())
].copy()

# Collect the unique set of proteins needed across all assemblies
all_proteins = set()
for proteins_str in high_score['Proteins']:
    all_proteins.update(proteins_str.split(' '))

print(f"High-score assemblies: {len(high_score)}")
print(f"Unique proteins to predict: {len(all_proteins)}")

## Step 1: Generate Predictions with ProtVS

Predict each required protein for the reference cell image and save
each result as a TIFF. Already-generated files are skipped so the
cell can be re-run safely. GPU memory is released after this step.

In [ ]:
ref_image = imread(REF_PATH)
print(f"Reference image shape: {ref_image.shape}")

model = ProtVS()

for protein in tqdm(sorted(all_proteins), desc="Predicting proteins"):
    out_path = os.path.join(PRED_DIR, f"{CELL_LINE}_{CELL_ID:03d}_{protein}_pred.tif")
    if os.path.exists(out_path):
        continue  # skip already generated

    result = model.predict(
        images=[ref_image],
        protein_names=[protein],
        cell_line_names=[CELL_LINE],
        num_inference_steps=50,
    )
    imwrite(out_path, result[0].astype(np.float32))

# Free GPU memory before visualization
del model
import torch; torch.cuda.empty_cache()

print(f"Predictions saved to {PRED_DIR}/")

## Step 2: Preprocessing Helpers

Utility functions used during visualization: contrast stretching,
coefficient-of-variation scoring for tile selection, and binarization.

In [ ]:
def autobright(ch, sat_lo=0.1, sat_hi=99.9, gamma=1.5):
    """Percentile-stretch + gamma correction → uint8."""
    lo, hi = np.percentile(ch, sat_lo), np.percentile(ch, sat_hi)
    if hi <= lo:
        hi = lo + 1
    return (np.clip((ch.astype(np.float32) - lo) / (hi - lo), 0, 1) ** gamma * 255).astype(np.uint8)

def snr_cv(ch):
    """Coefficient of variation — used to rank proteins by signal quality."""
    ch = ch.astype(np.float32)
    m  = ch.mean()
    return ch.std() / m if m > 1e-6 else 0.0

def make_tile(blue_u8, green_u8):
    """Compose a blue (DAPI) + green (protein) RGB tile."""
    t = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
    t[:, :, 2] = blue_u8
    t[:, :, 1] = green_u8
    return t

def sanitise(name):
    """Make a string safe for use as a filename."""
    return "".join(c for c in name if c.isalnum() or c in (' ', '-', '_')).rstrip()

def load_prediction(protein):
    """Load a saved prediction TIFF for the current cell; return None if missing."""
    path = os.path.join(PRED_DIR, f"{CELL_LINE}_{CELL_ID:03d}_{protein}_pred.tif")
    if not os.path.exists(path):
        return None
    return imread(path).astype(np.float32)[:TILE_SIZE, :TILE_SIZE]

# Pre-compute DAPI channel from the reference image
dapi_raw = imread(REF_PATH)[:TILE_SIZE, :TILE_SIZE, 2].astype(np.float32)
blue_ch  = autobright(dapi_raw)

# Figure geometry constants
tile_in  = DISPLAY_PX / DPI
title_h  = 0.32
xlabel_h = 0.24
n_cols   = N_REPRESENTATIVE + 2   # protein tiles + "+N" panel + heatmap
fig_w    = n_cols * tile_in + 0.05
fig_h    = tile_in + title_h + xlabel_h

print("Helpers ready.")

## Step 3: Visualize Assembly Maps

For each high-scoring assembly:
- Load the predicted images for all member proteins
- Rank by coefficient of variation and show the top-N as blue/green tiles
- Compute a per-pixel co-occupancy heatmap (fraction of proteins expressed above threshold)
- Save one PNG per assembly

In [ ]:
for _, row in tqdm(high_score.iterrows(), total=len(high_score), desc="Assemblies"):
    assembly_name = row['GPT4 default name']
    proteins      = row['Proteins'].split(' ')

    # Load predicted images for each protein in this assembly
    protein_imgs = {}
    for protein in proteins:
        img = load_prediction(protein)
        if img is not None:
            protein_imgs[protein] = img

    if not protein_imgs:
        print(f"  Skipping {assembly_name}: no predictions found")
        continue

    n_total     = len(protein_imgs)
    n_remaining = max(0, n_total - N_REPRESENTATIVE)

    # Rank proteins by signal quality; pick top-N for the tile strip
    scored       = {name: snr_cv(img) for name, img in protein_imgs.items()}
    top_proteins = sorted(scored, key=scored.get, reverse=True)[:N_REPRESENTATIVE]

    # Co-occupancy heatmap from all member proteins
    all_ch = np.stack(list(protein_imgs.values()), axis=2)
    binary = np.stack([
        (all_ch[:, :, c] > THRESHOLD_MULT * threshold_otsu(all_ch[:, :, c]))
        for c in range(all_ch.shape[2])
    ], axis=2)
    heatmap = binary.sum(axis=2) / all_ch.shape[2]

    # ── Figure layout ────────────────────────────────────────────
    fig = plt.figure(figsize=(fig_w, fig_h), facecolor='white')
    gs  = gridspec.GridSpec(
        1, n_cols, figure=fig,
        left=0.0, right=1.0,
        top=1.0 - title_h / fig_h,
        bottom=xlabel_h / fig_h,
        wspace=0.03,
    )

    # Protein tiles
    for col, protein in enumerate(top_proteins):
        ax = fig.add_subplot(gs[0, col])
        ax.set_facecolor('black')
        green_ch = autobright(protein_imgs[protein])
        ax.imshow(make_tile(blue_ch, green_ch), aspect='auto', interpolation='bilinear')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(True); sp.set_edgecolor('#cccccc'); sp.set_linewidth(0.6)
        ax.set_xlabel(protein, fontsize=FONT_PROTEIN, color='black',
                      fontfamily='DejaVu Sans', fontweight='bold', labelpad=3)

    # "+N more proteins" panel
    ax_more = fig.add_subplot(gs[0, N_REPRESENTATIVE])
    ax_more.set_facecolor('white')
    ax_more.set_xticks([]); ax_more.set_yticks([])
    for sp in ax_more.spines.values():
        sp.set_visible(False)
    if n_remaining > 0:
        ax_more.text(0.5, 0.55, f'+{n_remaining} more\nproteins...',
                     transform=ax_more.transAxes,
                     fontsize=8, fontweight='bold', color='#333333',
                     fontfamily='DejaVu Sans', ha='center', va='center')
    ax_more.set_xlabel('', fontsize=FONT_PROTEIN, labelpad=3)

    # Co-occupancy heatmap
    ax_hm = fig.add_subplot(gs[0, N_REPRESENTATIVE + 1])
    ax_hm.set_facecolor('black')
    ax_hm.imshow(heatmap, cmap='hot', vmin=0, vmax=1,
                 aspect='auto', interpolation='bilinear')
    ax_hm.set_xticks([]); ax_hm.set_yticks([])
    for sp in ax_hm.spines.values():
        sp.set_visible(True); sp.set_edgecolor('#cccccc'); sp.set_linewidth(0.6)
    ax_hm.set_xlabel('Assembly', fontsize=FONT_PROTEIN, color='#555555',
                     fontfamily='DejaVu Sans', labelpad=3, style='italic')

    # Title
    fig.suptitle(assembly_name, fontsize=FONT_TITLE, color='black', fontweight='bold',
                 fontfamily='DejaVu Sans',
                 x=0.5, y=1.0 - (title_h * 0.18) / fig_h, va='top')

    fig.savefig(os.path.join(OUT_DIR, f"{sanitise(assembly_name)}.png"),
                dpi=DPI, facecolor='white', bbox_inches='tight', pad_inches=0.04)
    plt.close(fig)

print(f"\nAssembly maps saved to {OUT_DIR}/")

## Step 4: Export Colorbar

Save a standalone vertical colorbar for the co-occupancy heatmap
so it can be composited with assembly panels in figures.

In [ ]:
fig_cb = plt.figure(figsize=(0.35, tile_in + xlabel_h), facecolor='white')
ax_cb  = fig_cb.add_axes([0.05, xlabel_h / (tile_in + xlabel_h),
                           0.40, tile_in  / (tile_in + xlabel_h)])
cb = fig_cb.colorbar(
    mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=0, vmax=1), cmap='hot'),
    cax=ax_cb, orientation='vertical',
)
cb.set_ticks([0, 0.5, 1])
cb.set_ticklabels(['0', '0.5', '1'], fontsize=FONT_CBAR, color='black')
cb.set_label('co-occupancy', fontsize=FONT_CBAR, color='#555555',
             fontfamily='DejaVu Sans', style='italic', labelpad=6)
cb.outline.set_edgecolor('#aaaaaa')
cb.outline.set_linewidth(0.5)
ax_cb.yaxis.set_tick_params(color='black', width=0.5, length=2)

fig_cb.savefig(os.path.join(OUT_DIR, '_colorbar.png'),
               dpi=DPI, facecolor='white', bbox_inches='tight', pad_inches=0.05)
plt.close(fig_cb)
print(f"Colorbar saved → {OUT_DIR}/_colorbar.png")

## Summary

**Outputs** (all under `assembly_map_plots/U2OS_<CELL_ID>/`):
- `<AssemblyName>.png` — protein tile strip + co-occupancy heatmap per assembly
- `_colorbar.png` — standalone colorbar for the heatmap panels

**Intermediate predictions** are cached in `assembly_map_predictions/U2OS_<CELL_ID>/`
as float32 TIFFs — re-running the notebook skips already-generated files.